# Gemma Edit-Discipline LoRA v2 — Colab training (A100)
Open via GitHub->Colab, set **A100 GPU** runtime, Run All.
**v2 (R11/R13):** ALL-LINEAR target modules (q/k/v/o + MLP) on the **46 all-repo** trajectories (23 rich + 23 fastapi). Replaces the v1 q/v-only / 23-rich adapter that gated 1/5.
RESUMABLE: checkpoints to Google Drive each epoch; re-run to add epochs. NF4 + grad-checkpointing + batch 1.

Spec: specs/gemma-lora-colab-training. Pre-registered: adapter must beat v19's 1/5 on a free Kaggle run before any scored slot. v6/0.12 SACRED.

In [ ]:
# 1. GPU assertion + pinned deps (E4). Stop if <40GB.
import torch, subprocess, sys
assert torch.cuda.is_available(), 'No GPU — set Runtime -> A100 GPU'
total = torch.cuda.mem_get_info()[1]/1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  {total:.0f}GB')
assert total >= 38, f'Need >=40GB A100 for 31B QLoRA; got {total:.0f}GB. Use a smaller base (27b/12b) or stop.'
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
                'transformers>=4.49','peft>=0.14','bitsandbytes>=0.45','accelerate>=1.3','safetensors','datasets'], check=True)
print('deps installed')

In [ ]:
# 2. Mount Drive (resumable checkpoints survive disconnect) + clone repo (code + trajectory data).
from google.colab import drive
drive.mount('/content/drive')
import os
# CACHE THE 62.6GB BASE ON DRIVE so it is downloaded ONCE, not every session.
# HF_HOME must be set BEFORE any transformers/huggingface_hub import. train_lora.py runs via `!python`
# (a subprocess) which inherits this os.environ, so setting it here covers the trainer too.
# First run still downloads ~63GB to Drive; every later run finds it cached (no re-download).
# NOTE: needs ~65GB free on Drive for the base + room for checkpoints. Check Drive space first.
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF cache (persisted to Drive):', os.environ['HF_HOME'])
# v2 uses a FRESH OUT dir — the widened all-linear adapter has DIFFERENT shapes than the v1 q/v-only
# checkpoint, so resuming the old dir would fail/mismatch. Train v2 from scratch into its own dir.
OUT = '/content/drive/MyDrive/gemma_lora/edit_discipline_v2_alllinear'
os.makedirs(OUT, exist_ok=True)
print('checkpoint dir (v2):', OUT)
!rm -rf /content/Gemma_Competition
!git clone --depth 1 https://github.com/GeorgeRCAdamJohnson/Gemma_Competition.git /content/Gemma_Competition
%cd /content/Gemma_Competition
!wc -l train/data/edit_discipline_all.jsonl

In [ ]:
# 3. HF auth for the gated bf16 base. Set HF_TOKEN in Colab Secrets (key icon, name=HF_TOKEN) OR paste below.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e:
    print('Set HF_TOKEN in Colab Secrets (recommended), or uncomment next line:')
    # os.environ['HF_TOKEN'] = 'hf_...'
from huggingface_hub import login
if os.environ.get('HF_TOKEN'): login(token=os.environ['HF_TOKEN'])
# E2 (measured): gemma-4-31b-it is PUBLIC (gated=False, 62.6GB) -> no token needed. 27b id is 401/missing.
# 80GB A100 holds the 31b bf16 base in NF4 easily. Train the TRUE 31b sibling (best transfer to W4A16-31b).
BASE = os.environ.get('LORA_BASE', 'google/gemma-4-31b-it')
print('training base:', BASE)
# Cache + Drive-space check: tells you if this run hits the cache or re-downloads ~63GB.
import shutil, glob
free_gb = shutil.disk_usage('/content/drive/MyDrive').free/1e9
cached = glob.glob(os.path.join(os.environ['HF_HOME'],'hub','models--google--gemma-4-31b-it','snapshots','*','*.safetensors'))
print(f'Drive free: {free_gb:.0f}GB  |  base cached on Drive: {"YES ("+str(len(cached))+" shards, no re-download)" if cached else "NO (first run downloads ~63GB to Drive)"}')
if not cached and free_gb < 70:
    print('WARNING: <70GB free and base not cached — the download may fill Drive. Free space or use /content (ephemeral) instead.')

In [ ]:
# 4a. SMOKE TEST (cheap, ~first-step only). Confirms the base's chat template accepts our
#     system/user/assistant/tool trajectory format BEFORE the full run. Trains 1 epoch to a THROWAWAY dir.
#     If this errors on apply_chat_template (tool role / multi assistant turn), we fix the data, not waste a full run.
!python train/train_lora.py \
  --out /content/_smoke \
  --base {BASE} \
  --r 32 --alpha 64 --maxlen 3072 --lr 2e-4 \
  --stage s1=data/edit_discipline_all.jsonl:1:2
print('\nSMOKE OK if you see per-step loss above (and the [lora] line shows ~290 ALL-LINEAR modules).')
print('If apply_chat_template errored, STOP and report.')

In [ ]:
# 4b. FULL TRAIN (resumable). Only run after 4a smoke passed. Checkpoints each epoch to OUT (Drive).
# v2: ALL-LINEAR adapter (set in train_lora.py fresh_adapter) on the 46 all-repo trajectories.
# r=32 (slice to 16/8 later via make_variant.py). Re-run to RESUME; raise :3 -> :5 to add epochs.
!python train/train_lora.py \
  --out {OUT} \
  --base {BASE} \
  --r 32 --alpha 64 --maxlen 3072 --lr 2e-4 \
  --stage s1=data/edit_discipline_all.jsonl:3

In [ ]:
# 5. Package the adapter: zip from Drive + print sha256/size for the manifest. Download via the Files pane,
#    or it's already on your Drive at OUT. This adapter dir drops into submit_v19_lora/adapters/.
import os, hashlib, glob
files = [f for f in glob.glob(OUT+'/*') if os.path.isfile(f)]
print('adapter files:')
for f in files:
    h = hashlib.sha256(open(f,'rb').read()).hexdigest()[:16]
    print(f'  {os.path.basename(f):30s} {os.path.getsize(f)/1e6:7.1f}MB  sha256={h}')
!cd {OUT} && zip -qr /content/edit_discipline_v2.zip . && ls -la /content/edit_discipline_v2.zip
print('\nAdapter is on Drive at', OUT, 'and zipped at /content/edit_discipline_v2.zip (download from Files pane).')